# 강의 03 · 실습 3 — RAG 검색기 구축 · (3) 변형

## 1. 문제상황

- 시립도서관 안내 데스크에는 이용안내·대출·시설에 관한 질문이 들어오며, FAQ는 12개 항목으로 정리되어 있습니다.
- 손님은 「예약은 어떻게 해요」처럼 짧게 묻고, 담당자는 어느 카테고리의 어느 항목이 답인지 찾아 줍니다.
- 손님이 대출 창구에서 물었으니 대출 카테고리 안의 답을 원하는데도, 검색은 카테고리를 보지 않아 회원 가입 같은 다른 카테고리의 비슷한 문장이 섞여 나옵니다.
- 저장소를 만든 뒤 프로그램을 껐다 켜면 다시 만들어야 하는지, 디스크에 남아 있는지도 확인해야 합니다.

## 2. 문제와 목표

- **문제**: 문서가 바뀌면 검색기를 다시 세워야 하고, 카테고리를 알고 있어도 검색이 그 안에서만 찾지 않으며, 저장소가 디스크에 남아 있는지 확인하지 않습니다.
- **목표**: 도서관 FAQ 12행을 적재해 검색기를 세우고, 카테고리 필터를 건 검색과 걸지 않은 검색을 비교하며, 저장소를 디스크에서 다시 열어 항목 수를 확인하고, 문서 밖 질문은 임계값으로 컷하는 검색기를 만듭니다.
    - 도서관 FAQ: `library_faq.csv`(12행), 카테고리는 이용안내·대출·시설. 저장 디렉터리: `chroma_library`.
    - 카테고리 필터: 메타데이터 `category` 값이 같은 청크 안에서만 검색하는 조건.
    - 임계값 1.5와 고정 안내 문장: 「문서에서 근거를 찾지 못했습니다. 안내 창구로 문의해 주세요.」
- **목표 달성 여부의 판정 기준**: 적재 문서 수가 12이고, 「예약은 어떻게 하나요?」를 필터 없이 검색하면 상위 3개에 다른 카테고리 청크(chunk)가 섞이지만 카테고리 필터를 걸면 상위 3개가 모두 대출 카테고리이며, 디스크에서 다시 연 저장소의 항목 수가 12이고, 문서 밖 질문이 컷되는 것을 실행 기록에서 확인합니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec03_ex03_s3_diagram.svg)

## 4. 단계별 요구사항

1. **문서를 적재합니다.**
    - `library_faq.csv`를 `csv.DictReader`로 읽고(`utf-8-sig`), `Question`이 빈 행은 버리고, 행마다 `[카테고리] Q: 질문\nA: 답변` 본문과 `{"row": 행 번호, "category": 카테고리}` 메타데이터를 가진 `Document`를 만듭니다.
    - 적재 문서 수(12)를 출력합니다.
2. **임베딩을 준비합니다.**
    - `OpenAIEmbeddings(model="text-embedding-3-small")`로 임베딩 부품 `emb`를 만들고, 문장 하나를 벡터로 바꿔 벡터의 길이를 출력합니다.
3. **저장소를 구축하고 영속한 뒤 다시 엽니다.**
    - `persist_directory="chroma_library"`와 `ids`(`row-<행 번호>`)로 저장소를 만들고 항목 수를 출력합니다.
    - 그 다음 `Chroma(persist_directory="chroma_library", embedding_function=emb)`로 디스크에서 다시 열어 항목 수를 출력합니다.
    - 두 수가 같아야 합니다.
    - 출력 줄은 「구축 직후 항목 수」와 「디스크에서 다시 연 항목 수」로 표시합니다.
4. **필터를 걸어 검색합니다.**
    - 「예약은 어떻게 하나요?」를 필터 없이 `k=3`으로 검색한 결과와, `filter={"category": "대출"}`을 걸어 검색한 결과를 차례로 출력합니다.
    - 필터 없는 결과에는 다른 카테고리의 청크가 섞이고, 필터를 건 결과는 상위 청크가 모두 `[대출]`이어야 합니다.
5. **임계값으로 컷합니다.**
    - `THRESHOLD = 1.5`와 고정 안내 문장 `NO_EVIDENCE = "문서에서 근거를 찾지 못했습니다. 안내 창구로 문의해 주세요."`를 두고, 1위 점수가 임계값 이하이면 그 청크의 본문을, 넘으면 `NO_EVIDENCE`를 돌려주는 함수 `answer_or_cut`을 만들어, 문서 안 질문(「주차 되나요?」)과 문서 밖 질문(「파이썬 리스트 정렬은 어떻게 하나요?」)을 넣어 점수·판정·결과를 출력합니다.
    - 판정은 「통과」 또는 「컷」으로 표시합니다.

## 5. 코드 골격 — RAG 인덱싱·검색 5단

다섯 단계는 놀이공원 FAQ 검색기와 같고, ③에 다시 열기, ④에 메타데이터 필터가 더해집니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 문서 적재 | 원본 파일을 읽어 검색 단위 문서로 만듭니다 | `Document(page_content=..., metadata=...)` | 1 |
| ② 임베딩 준비 | 문장을 숫자 벡터로 바꿀 모델을 지정합니다 | `OpenAIEmbeddings(model="text-embedding-3-small")` | 2 |
| ③ 저장소 구축·영속 | 문서와 임베딩을 넣어 저장소를 만들고 디렉터리에 남깁니다 | `Chroma.from_documents(...)`, `Chroma(persist_directory=..., embedding_function=emb)` | 3 |
| ④ 점수 동반 검색 | 질문을 넣어 가까운 문서와 그 거리 점수를 함께 받습니다 | `db.similarity_search_with_score(q, k=3, filter={"category": ...})` | 4 |
| ⑤ 임계값 컷 | 점수가 기준을 넘으면 문서 근거를 쓰지 않고 다른 경로로 보냅니다 | `if s <= THRESHOLD` | 5 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 API 키를 읽습니다. 임베딩 모델도 OpenAI API를 쓰므로 같은 키가 필요합니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

In [ ]:
import csv
import os

from dotenv import load_dotenv, find_dotenv

from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")
print("준비를 마쳤습니다.")

### 단계 ① — 문서 적재 (요구사항 1)

- 검색 단위는 `Document`입니다. 본문(`page_content`)이 임베딩되어 검색에 쓰이고, 메타데이터(`metadata`)는 청크와 함께 돌아와 출처 표시나 필터에 쓰입니다.
- FAQ 한 행을 청크 하나로 삼습니다. 질문과 답을 한 본문에 넣어야 질문 표현으로 검색해도 답이 함께 돌아옵니다.

In [ ]:
# 여기에 단계 ①(library_faq.csv 적재)을 작성합니다.

### 단계 ② — 임베딩 준비 (요구사항 2)

- 임베딩은 문장을 숫자 벡터로 바꾸는 모델입니다. 의미가 가까운 문장은 벡터도 가깝습니다.
- 문서를 넣을 때와 질문을 넣을 때 같은 임베딩을 써야 같은 좌표계에서 거리를 잴 수 있습니다.

In [ ]:
# 여기에 단계 ②(임베딩 준비)를 작성합니다.

### 단계 ③ — 저장소 구축·영속 (요구사항 3)

- `Chroma.from_documents`가 문서마다 임베딩을 계산해 저장소에 넣습니다. `persist_directory`를 주면 디렉터리에 남아 프로그램이 끝나도 유지됩니다.
- 문서 id를 행 번호로 주면 같은 셀을 다시 실행해도 같은 id에 덮어써 항목이 늘지 않습니다.
- 다시 열 때는 문서를 넣지 않습니다. 디렉터리에 남은 항목을 그대로 씁니다.

In [ ]:
# 여기에 단계 ③(구축·영속·다시 열기)을 작성합니다.

### 단계 ④ — 점수 동반 검색 (요구사항 4)

- 필터는 메타데이터의 키와 값으로 겁니다. 필터를 걸면 저장소는 그 청크들 안에서만 거리를 잽니다.
- 같은 질문을 필터 없이·필터 걸고 두 번 검색해 상위 청크의 카테고리를 비교합니다.

In [ ]:
# 여기에 단계 ④(필터 없이·필터 걸고 검색)를 작성합니다.

### 단계 ⑤ — 임계값 컷 (요구사항 5)

- 임계값은 문서 안 질문의 점수 분포와 문서 밖 질문의 점수 분포 사이에 긋는 선입니다. 여기서는 1.5를 씁니다.
- 컷된 질문을 보낼 곳을 정해야 설계가 닫힙니다. 이 실습에서는 고정 안내 문장으로 보냅니다.

In [ ]:
# 여기에 단계 ⑤(임계값 컷 함수와 두 질문 실행)를 작성합니다.

## 7. 실행 결과 확인

셀을 위에서 아래로 모두 실행한 뒤 다음 세 가지를 확인합니다.

1. 단계 ①의 「적재 문서 수」가 12이고, 단계 ③의 「구축 직후 항목 수」와 「디스크에서 다시 연 항목 수」가 모두 12입니다.
2. 단계 ④에서 필터 없는 검색의 상위 3개에는 `[이용안내]` 청크가 섞이고, 필터를 건 검색의 상위 3개는 모두 `[대출]`로 시작하며, 각 청크에 `A:` 답변이 들어 있습니다.
3. 단계 ⑤에서 「주차 되나요?」는 「통과」와 주차 안내 청크가, 「파이썬 리스트 정렬은 어떻게 하나요?」는 「컷」과 고정 안내 문장이 찍힙니다.

세 가지가 모두 확인되면 완성입니다.